In [1]:
from google.colab import files
uploaded = files.upload()

Saving cardio_train.csv to cardio_train.csv


In [2]:
import csv
file_path = "cardio_train.csv"

data = []

with open(file_path, mode="r", newline="", encoding="utf-8") as file:
  reader = csv.DictReader(file, delimiter=";")

  for row in reader:
    row["id"] = int(row["id"])
    row["age"] = int(row["age"])
    row["gender"] = int(row["gender"])
    row["height"] = int(row["height"])
    row["weight"] = float(row["weight"])
    row["ap_hi"] = int(row["ap_hi"])
    row["ap_lo"] = int(row["ap_lo"])
    row["cholesterol"] = int(row["cholesterol"])
    row["gluc"] = int(row["gluc"])
    row["smoke"] = int(row["smoke"])
    row["alco"] = int(row["alco"])
    row["active"] = int(row["active"])
    row["cardio"] = int(row["cardio"])

    row["age_years"] = row["age"] / 365
    height_m = row["height"] / 100
    row["bmi"] = row["weight"] / (height_m ** 2)

    data.append(row)

print("Jumlah data:", len(data))
print("Contoh data pertama:")
print(data[0])

Jumlah data: 70000
Contoh data pertama:
{'id': 0, 'age': 18393, 'gender': 2, 'height': 168, 'weight': 62.0, 'ap_hi': 110, 'ap_lo': 80, 'cholesterol': 1, 'gluc': 1, 'smoke': 0, 'alco': 0, 'active': 1, 'cardio': 0, 'age_years': 50.391780821917806, 'bmi': 21.9671201814059}


In [3]:
clean_data = []
for row in data:
  if (80 <= row["ap_hi"] <= 240 and
      40 <= row["ap_lo"] <= 160 and
      row["ap_hi"] > row["ap_lo"] and
      120 <= row["height"] <= 220 and
      30 <= row["weight"] <= 200):
    clean_data.append(row)

print("Jumlah data setelah dibersihkan:", len(clean_data))
print(clean_data[0])

Jumlah data setelah dibersihkan: 68608
{'id': 0, 'age': 18393, 'gender': 2, 'height': 168, 'weight': 62.0, 'ap_hi': 110, 'ap_lo': 80, 'cholesterol': 1, 'gluc': 1, 'smoke': 0, 'alco': 0, 'active': 1, 'cardio': 0, 'age_years': 50.391780821917806, 'bmi': 21.9671201814059}


In [4]:
# fungsi keanggotaan dasar

def turun(x, a, b):
  if x <= a:
    return 1
  elif x >= b:
    return 0
  else:
    return (b - x) / (b - a)

def naik(x, a, b):
  if x <= a:
    return 0
  elif x >= b:
    return 1
  else:
    return (x - a) / (b - a)

def segitiga(x, a, b, c):
  if x <= a or x >= c:
    return 0
  elif x == b:
    return 1
  elif a < x < b:
    return (x - a) / (b - a)
  else:
    return (c - x) / (c - b)

In [5]:
# membership input
# umur
def umur_muda(age):
  return turun(age, 35, 45)

def umur_dewasa(age):
  return segitiga(age, 35, 50, 65)

def umur_lansia(age):
  return naik(age, 55, 65)

# bmi
def bmi_normal(bmi):
  return turun(bmi, 23, 27)

def bmi_overweight(bmi):
  return segitiga(bmi, 25, 30, 35)

def bmi_obesitas(bmi):
  return naik(bmi, 30, 40)

# tekanan darah sistolik / ap_hi
def sistolik_normal(ap_hi):
  return turun(ap_hi, 120, 140)

def sistolik_tinggi(ap_hi):
  return segitiga(ap_hi, 130, 150, 170)

def sistolik_sangat_tinggi(ap_hi):
  return naik(ap_hi, 160, 180)

#tekanan darah distolik / ap_lo
def distolik_normal(ap_lo):
  return turun(ap_lo, 80, 90)

def distolik_tinggi(ap_lo):
  return segitiga(ap_lo, 85, 100, 115)

def distolik_sangat_tinggi(ap_lo):
  return naik(ap_lo, 105, 120)

# kolesterol
def kolesterol_normal(chol):
  if chol == 1:
    return 1
  return 0

def kolesterol_tinggi(chol):
  if chol == 2:
    return 1
  return 0.5 if chol == 3 else 0

def kolesterol_sangat_tinggi(chol):
  if chol == 3:
    return 1
  return 0

In [6]:
# fuzzy mamdani
def mamdani(row):

    age = row["age_years"]
    bmi = row["bmi"]
    ap_hi = row["ap_hi"]
    chol = row["cholesterol"]

    rules = []

    # Rule 1
    # IF umur muda AND BMI normal AND tekanan normal AND kolesterol normal
    # THEN risiko rendah
    r1 = min(
        umur_muda(age),
        bmi_normal(bmi),
        tekanan_normal(ap_hi),
        kolesterol_normal(chol)
    )
    rules.append(("rendah", r1))

    # Rule 2
    # IF umur muda AND tekanan normal
    # THEN risiko rendah
    r2 = min(
        umur_muda(age),
        tekanan_normal(ap_hi)
    )
    rules.append(("rendah", r2))

    # Rule 3
    # IF BMI normal AND kolesterol normal
    # THEN risiko rendah
    r3 = min(
        bmi_normal(bmi),
        kolesterol_normal(chol)
    )
    rules.append(("rendah", r3))

    # Rule 4
    # IF umur dewasa AND BMI normal AND tekanan normal
    # THEN risiko rendah
    r4 = min(
        umur_dewasa(age),
        bmi_normal(bmi),
        tekanan_normal(ap_hi)
    )
    rules.append(("rendah", r4))

    # Rule 5
    # IF umur muda AND BMI overweight AND tekanan normal
    # THEN risiko rendah
    r5 = min(
        umur_muda(age),
        bmi_overweight(bmi),
        tekanan_normal(ap_hi)
    )
    rules.append(("rendah", r5))

    # Rule 6
    # IF umur dewasa AND BMI overweight AND tekanan tinggi
    # THEN risiko sedang
    r6 = min(
        umur_dewasa(age),
        bmi_overweight(bmi),
        tekanan_tinggi(ap_hi)
    )
    rules.append(("sedang", r6))

    # Rule 7
    # IF kolesterol tinggi AND tekanan normal
    # THEN risiko sedang
    r7 = min(
        kolesterol_tinggi(chol),
        tekanan_normal(ap_hi)
    )
    rules.append(("sedang", r7))

    # Rule 8
    # IF umur dewasa AND kolesterol tinggi
    # THEN risiko sedang
    r8 = min(
        umur_dewasa(age),
        kolesterol_tinggi(chol)
    )
    rules.append(("sedang", r8))

    # Rule 9
    # IF BMI overweight AND tekanan normal
    # THEN risiko sedang
    r9 = min(
        bmi_overweight(bmi),
        tekanan_normal(ap_hi)
    )
    rules.append(("sedang", r9))

    # Rule 10
    # IF umur lansia AND tekanan normal
    # THEN risiko sedang
    r10 = min(
        umur_lansia(age),
        tekanan_normal(ap_hi)
    )
    rules.append(("sedang", r10))

    # Rule 11
    # IF BMI overweight AND kolesterol tinggi
    # THEN risiko sedang
    r11 = min(
        bmi_overweight(bmi),
        kolesterol_tinggi(chol)
    )
    rules.append(("sedang", r11))

    # Rule 12
    # IF umur dewasa AND tekanan tinggi
    # THEN risiko sedang
    r12 = min(
        umur_dewasa(age),
        tekanan_tinggi(ap_hi)
    )
    rules.append(("sedang", r12))

    # Rule 13
    # IF kolesterol sangat tinggi AND tekanan normal
    # THEN risiko sedang
    r13 = min(
        kolesterol_sangat_tinggi(chol),
        tekanan_normal(ap_hi)
    )
    rules.append(("sedang", r13))

    # Rule 14
    # IF umur lansia AND tekanan sangat tinggi
    # THEN risiko tinggi
    r14 = min(
        umur_lansia(age),
        tekanan_sangat_tinggi(ap_hi)
    )
    rules.append(("tinggi", r14))

    # Rule 15
    # IF BMI obesitas AND tekanan tinggi
    # THEN risiko tinggi
    r15 = min(
        bmi_obesitas(bmi),
        tekanan_tinggi(ap_hi)
    )
    rules.append(("tinggi", r15))

    # Rule 16
    # IF kolesterol sangat tinggi AND tekanan tinggi
    # THEN risiko tinggi
    r16 = min(
        kolesterol_sangat_tinggi(chol),
        tekanan_tinggi(ap_hi)
    )
    rules.append(("tinggi", r16))

    # Rule 17
    # IF umur lansia AND BMI obesitas
    # THEN risiko tinggi
    r17 = min(
        umur_lansia(age),
        bmi_obesitas(bmi)
    )
    rules.append(("tinggi", r17))

    # Rule 18
    # IF umur lansia AND kolesterol sangat tinggi
    # THEN risiko tinggi
    r18 = min(
        umur_lansia(age),
        kolesterol_sangat_tinggi(chol)
    )
    rules.append(("tinggi", r18))

    # Rule 19
    # IF BMI obesitas AND kolesterol sangat tinggi
    # THEN risiko tinggi
    r19 = min(
        bmi_obesitas(bmi),
        kolesterol_sangat_tinggi(chol)
    )
    rules.append(("tinggi", r19))

    # Rule 20
    # IF umur lansia AND tekanan tinggi AND kolesterol tinggi
    # THEN risiko tinggi
    r20 = min(
        umur_lansia(age),
        tekanan_tinggi(ap_hi),
        kolesterol_tinggi(chol)
    )
    rules.append(("tinggi", r20))

    # Nilai crisp sederhana untuk defuzzifikasi
    nilai_output = {
        "rendah": 25,
        "sedang": 55,
        "tinggi": 85
    }

    pembilang = 0
    penyebut = 0

    for kategori, alpha in rules:
        pembilang += alpha * nilai_output[kategori]
        penyebut += alpha

    if penyebut == 0:
        return 0

    return pembilang / penyebut

In [7]:
# fuzzy sugeno
def sugeno(row):

    age = row["age_years"]
    bmi = row["bmi"]
    ap_hi = row["ap_hi"]
    chol = row["cholesterol"]

    rules = []

    # Rule 1
    r1 = min(
        umur_muda(age),
        bmi_normal(bmi),
        tekanan_normal(ap_hi),
        kolesterol_normal(chol)
    )
    rules.append((r1, 20))

    # Rule 2
    r2 = min(
        umur_muda(age),
        tekanan_normal(ap_hi)
    )
    rules.append((r2, 25))

    # Rule 3
    r3 = min(
        bmi_normal(bmi),
        kolesterol_normal(chol)
    )
    rules.append((r3, 20))

    # Rule 4
    r4 = min(
        umur_dewasa(age),
        bmi_normal(bmi),
        tekanan_normal(ap_hi)
    )
    rules.append((r4, 30))

    # Rule 5
    r5 = min(
        umur_muda(age),
        bmi_overweight(bmi),
        tekanan_normal(ap_hi)
    )
    rules.append((r5, 35))

    # Rule 6
    r6 = min(
        umur_dewasa(age),
        bmi_overweight(bmi),
        tekanan_tinggi(ap_hi)
    )
    rules.append((r6, 55))

    # Rule 7
    r7 = min(
        kolesterol_tinggi(chol),
        tekanan_normal(ap_hi)
    )
    rules.append((r7, 50))

    # Rule 8
    r8 = min(
        umur_dewasa(age),
        kolesterol_tinggi(chol)
    )
    rules.append((r8, 55))

    # Rule 9
    r9 = min(
        bmi_overweight(bmi),
        tekanan_normal(ap_hi)
    )
    rules.append((r9, 50))

    # Rule 10
    r10 = min(
        umur_lansia(age),
        tekanan_normal(ap_hi)
    )
    rules.append((r10, 60))

    # Rule 11
    r11 = min(
        bmi_overweight(bmi),
        kolesterol_tinggi(chol)
    )
    rules.append((r11, 60))

    # Rule 12
    r12 = min(
        umur_dewasa(age),
        tekanan_tinggi(ap_hi)
    )
    rules.append((r12, 55))

    # Rule 13
    r13 = min(
        kolesterol_sangat_tinggi(chol),
        tekanan_normal(ap_hi)
    )
    rules.append((r13, 65))

    # Rule 14
    r14 = min(
        umur_lansia(age),
        tekanan_sangat_tinggi(ap_hi)
    )
    rules.append((r14, 90))

    # Rule 15
    r15 = min(
        bmi_obesitas(bmi),
        tekanan_tinggi(ap_hi)
    )
    rules.append((r15, 85))

    # Rule 16
    r16 = min(
        kolesterol_sangat_tinggi(chol),
        tekanan_tinggi(ap_hi)
    )
    rules.append((r16, 90))

    # Rule 17
    r17 = min(
        umur_lansia(age),
        bmi_obesitas(bmi)
    )
    rules.append((r17, 85))

    # Rule 18
    r18 = min(
        umur_lansia(age),
        kolesterol_sangat_tinggi(chol)
    )
    rules.append((r18, 90))

    # Rule 19
    r19 = min(
        bmi_obesitas(bmi),
        kolesterol_sangat_tinggi(chol)
    )
    rules.append((r19, 90))

    # Rule 20
    r20 = min(
        umur_lansia(age),
        tekanan_tinggi(ap_hi),
        kolesterol_tinggi(chol)
    )
    rules.append((r20, 85))

    pembilang = 0
    penyebut = 0

    for alpha, z in rules:
        pembilang += alpha * z
        penyebut += alpha

    if penyebut == 0:
        return 0

    return pembilang / penyebut

In [8]:
# test hasil fuzzy
for i in range(5):
    row = clean_data[i]

    hasil_mamdani = mamdani(row)
    hasil_sugeno = sugeno(row)

    print("Data ke-", i + 1)
    print("Umur:", round(row["age_years"], 2))
    print("BMI:", round(row["bmi"], 2))
    print("Sistolik:", row["ap_hi"])
    print("Kolesterol:", row["cholesterol"])
    print("Label asli cardio:", row["cardio"])
    print("Hasil Mamdani:", round(hasil_mamdani, 2))
    print("Hasil Sugeno:", round(hasil_sugeno, 2))
    print("-" * 40)

Data ke- 1
Umur: 50.39
BMI: 21.97
Sistolik: 110
Kolesterol: 1
Label asli cardio: 0
Hasil Mamdani: 25.0
Hasil Sugeno: 24.93
----------------------------------------
Data ke- 2
Umur: 55.42
BMI: 34.93
Sistolik: 140
Kolesterol: 3
Label asli cardio: 1
Hasil Mamdani: 73.31
Hasil Sugeno: 75.3
----------------------------------------
Data ke- 3
Umur: 51.66
BMI: 23.51
Sistolik: 130
Kolesterol: 3
Label asli cardio: 1
Hasil Mamdani: 47.5
Hasil Sugeno: 50.0
----------------------------------------
Data ke- 4
Umur: 48.28
BMI: 28.71
Sistolik: 150
Kolesterol: 1
Label asli cardio: 1
Hasil Mamdani: 55.0
Hasil Sugeno: 55.0
----------------------------------------
Data ke- 5
Umur: 47.87
BMI: 23.01
Sistolik: 100
Kolesterol: 1
Label asli cardio: 0
Hasil Mamdani: 25.0
Hasil Sugeno: 24.63
----------------------------------------


In [9]:
# kategori risiko
def kategori_risiko(nilai):
    if nilai < 40:
        return "Rendah"
    elif nilai < 70:
        return "Sedang"
    else:
        return "Tinggi"

In [10]:
for i in range(5):
    row = clean_data[i]

    hasil_mamdani = mamdani(row)
    hasil_sugeno = sugeno(row)

    print("Data ke-", i + 1)
    print("Hasil Mamdani:", round(hasil_mamdani, 2), "-", kategori_risiko(hasil_mamdani))
    print("Hasil Sugeno:", round(hasil_sugeno, 2), "-", kategori_risiko(hasil_sugeno))
    print("Label asli cardio:", row["cardio"])
    print("-" * 40)

Data ke- 1
Hasil Mamdani: 25.0 - Rendah
Hasil Sugeno: 24.93 - Rendah
Label asli cardio: 0
----------------------------------------
Data ke- 2
Hasil Mamdani: 73.31 - Tinggi
Hasil Sugeno: 75.3 - Tinggi
Label asli cardio: 1
----------------------------------------
Data ke- 3
Hasil Mamdani: 47.5 - Sedang
Hasil Sugeno: 50.0 - Sedang
Label asli cardio: 1
----------------------------------------
Data ke- 4
Hasil Mamdani: 55.0 - Sedang
Hasil Sugeno: 55.0 - Sedang
Label asli cardio: 1
----------------------------------------
Data ke- 5
Hasil Mamdani: 25.0 - Rendah
Hasil Sugeno: 24.63 - Rendah
Label asli cardio: 0
----------------------------------------


In [11]:
#evaluasi sederhana
def prediksi_cardio(nilai_risiko):
    if nilai_risiko >= 50:
        return 1
    else:
        return 0


benar_mamdani = 0
benar_sugeno = 0

jumlah_data = len(clean_data)

for row in clean_data:
    hasil_mamdani = mamdani(row)
    hasil_sugeno = sugeno(row)

    pred_mamdani = prediksi_cardio(hasil_mamdani)
    pred_sugeno = prediksi_cardio(hasil_sugeno)

    if pred_mamdani == row["cardio"]:
        benar_mamdani += 1

    if pred_sugeno == row["cardio"]:
        benar_sugeno += 1

akurasi_mamdani = benar_mamdani / jumlah_data * 100
akurasi_sugeno = benar_sugeno / jumlah_data * 100

print("Akurasi Mamdani:", round(akurasi_mamdani, 2), "%")
print("Akurasi Sugeno:", round(akurasi_sugeno, 2), "%")

Akurasi Mamdani: 63.44 %
Akurasi Sugeno: 63.79 %
